# Notebook 4 — LLM-as-a-Judge con mitigación de sesgos (alineado con S06)
**Módulo M2 — Evaluación de NER clínico**

Este notebook implementa la dimensión de evaluación cualitativa con un **LLM como juez clínico**, complementando las métricas de exact-match (Dimensión 1) y similitud semántica (Dimensión 2), siguiendo fielmente lo enseñado en S06:

1. **Carga del Gold Set** (`gold_examples.jsonl`) con el modelo fine-tuneado (`clinical_BERT + LoRA`).
2. **Rich Examples** con el campo `criterio` de cada caso (que antes el harness ignoraba).
3. **Juez LLM (`openai/gpt-oss-120b`)** con rúbrica 1–5 anclada y **veredicto binario de dominio** (`cumple_criterio`: si/no), la tercera dimensión del harness de S06.
4. **Mitigación rigurosa de sesgos (S06, Lab B):**
   - **Sesgo de posición:** comparación **pairwise A/B con intercambio de orden**. No se intercambian gold y predicción dentro de la rúbrica. El test es un diagnóstico y **no se promedia con el score final**.
   - **Sesgo de longitud:** pares controlados de respuesta larga **parcialmente correcta** vs. corta correcta.
   - **Sesgo de auto-preferencia:** anonimización total del output y uso de familia externa.
5. **Scorecard y triangulación con F1 exacto:** detección de errores de span vs errores de comprensión clínica.

El score final del juez es el **pointwise** (una sola llamada por documento); el test de posición solo diagnostica y no contamina ese score.



## 0. Setup e imports

In [3]:
try:
    import google.colab
    IN_COLAB = True
    from google.colab import drive
    drive.mount("/content/drive")
    print("Ejecutando en Google Colab — Drive montado.")
except ImportError:
    IN_COLAB = False
    print("Ejecutando en entorno local.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ejecutando en Google Colab — Drive montado.


In [4]:
import subprocess, sys

required = ["groq", "datasets", "peft", "transformers", "accelerate"]
installed = []

for pkg in required:
    try:
        __import__(pkg)
    except ImportError:
        installed.append(pkg)

if installed:
    print(f"Instalando: {installed}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + installed)
    print("Instalación completada.")
else:
    print("Todas las dependencias ya están disponibles.")
# Desinstalar torchao si existe para evitar el bug de versión con peft en Colab
try:
    subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
except Exception:
    pass


Todas las dependencias ya están disponibles.


In [5]:
import re, json, random, time
from pathlib import Path
import itertools
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from peft import PeftModel

from groq import Groq

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Seeds fijadas (SEED={SEED}). Device: {device}")

import transformers as _tr, peft as _peft
print(f"  transformers: {_tr.__version__}")
print(f"  peft:         {_peft.__version__}")
print(f"  torch:        {torch.__version__}")
print(f"  numpy:        {np.__version__}")
# Evitar incompatibilidad de versión de torchao en Colab
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass


Seeds fijadas (SEED=42). Device: cuda
  transformers: 5.16.1
  peft:         0.20.0
  torch:        2.11.0+cu128
  numpy:        2.1.3


In [ ]:
# ============================================================
#  CONFIGURACIÓN DEL PROYECTO (Rutas sincronizadas con el harness)
# ============================================================

if IN_COLAB:
    _drive_root = Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud")
    if not _drive_root.exists():
        _drive_root = Path("/content/drive/MyDrive/TopicosIA")
    PROJECT_ROOT = _drive_root
else:
    PROJECT_ROOT = Path("..") if Path("../M1").exists() else Path(".")

# Adaptador LoRA de Clinical BERT
MODEL_DIR = PROJECT_ROOT / "M1" / "saved_models" / "clinical_bert-distemist-lora"

# Gold set generado para evaluación de harness (formato JSONL)
_gold_candidates = [
    PROJECT_ROOT / "M2" / "eval_harness" / "gold_examples.jsonl",
    PROJECT_ROOT / "M2" / "gold_examples.jsonl",
    PROJECT_ROOT / "eval_harness" / "gold_examples.jsonl",
    PROJECT_ROOT / "gold_examples.jsonl",
    Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/eval_harness/gold_examples.jsonl"),
    Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/gold_examples.jsonl"),
    Path("/content/drive/MyDrive/TopicosIA/M2/eval_harness/gold_examples.jsonl"),
    Path("/content/drive/MyDrive/TopicosIA/M2/gold_examples.jsonl"),
    Path("/content/drive/MyDrive/Topicos-IA/M2/eval_harness/gold_examples.jsonl"),
    Path("/content/drive/MyDrive/Topicos-IA/M2/gold_examples.jsonl"),
    Path("M2/eval_harness/gold_examples.jsonl"),
    Path("M2/gold_examples.jsonl"),
    Path("gold_examples.jsonl"),
]
GOLD_SET_PATH = next((p for p in _gold_candidates if p.exists()), PROJECT_ROOT / "M2" / "gold_examples.jsonl")

# Directorio de salidas de M2
OUTPUT_DIR = PROJECT_ROOT / "M2" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ruta al resultado de dimensión 1 (exact match)
DIM1_PATH = OUTPUT_DIR / "resultado_dimension1.json"

# Checkpoint base oficial del proyecto
BASE_CHECKPOINT = "PlanTL-GOB-ES/roberta-base-biomedical-clinical-es"

# ============================================================
#  CONFIGURACIÓN DEL JUEZ LLM (Groq)
# ============================================================
# Clave 1 (obligatoria). Se puede pegar aqui o configurar en Colab Secrets
# como GROQ_API_KEY. Clave 2 opcional (GROQ_API_KEY_2): suma cupo diario.
GROQ_API_KEY   = ""   # gratis en https://console.groq.com/keys
GROQ_API_KEY_2 = ""   # opcional: segunda cuenta, suma 200k tokens/dia

# Modelo juez (reproducibilidad unificada con el harness).
# Se usa qwen/qwen3.8-27b en vez de openai/gpt-oss-120b porque este ultimo es un
# modelo de razonamiento que gasta ~1000-1300 tokens internos por llamada y no
# alcanza el cupo gratuito de Groq (200k tokens/dia) para 59 docs x 3 llamadas.
# qwen3.8-27b resuelve la misma tarea con ~100 tokens por llamada.
JUDGE_MODEL = "qwen/qwen3.8-27b"

JUDGE_TEMPERATURE = 0.0   # determinístico
JUDGE_MAX_TOKENS  = 600   # la respuesta real usa ~100-200 tokens

# Pausa entre llamadas: Groq limita por minuto (TPM/OTPM/RPM) y una ráfaga sin
# pausa los agota (429 en cascada). 3s mantiene el ritmo bajo el cupo gratuito.
JUDGE_PAUSA_ENTRE_LLAMADAS = 3.0

# Parámetros de chunking (M1)
WINDOW_WORDS  = 277
OVERLAP_WORDS = 50

print(f"PROJECT_ROOT:  {PROJECT_ROOT}")
print(f"MODEL_DIR:     {MODEL_DIR}")
print(f"GOLD_SET_PATH: {GOLD_SET_PATH} (existe: {GOLD_SET_PATH.exists()})")
print(f"OUTPUT_DIR:    {OUTPUT_DIR}")
print(f"JUDGE_MODEL:   {JUDGE_MODEL}")

# Si el adaptador no existe aún en local/drive, soporte de descarga directa desde la rama
if not (MODEL_DIR / "adapter_model.safetensors").exists():
    print(f"[INFO] Modelo LoRA no encontrado en {MODEL_DIR}. Intentando clonar desde rama clinical-bert...")
    try:
        import subprocess, shutil
        MODEL_DIR.mkdir(parents=True, exist_ok=True)
        subprocess.run([
            "git", "clone", "--depth", "1", "-b", "clinical-bert",
            "https://github.com/luisNP21/Topicos-IA.git", "/tmp/repo_clinical"
        ], check=True)
        for item in Path("/tmp/repo_clinical/M1/clinical_BERT/clinical_bert-distemist-lora").glob("*"):
            shutil.copy2(item, MODEL_DIR)
        print(f"[OK] Modelo LoRA descargado y listo en {MODEL_DIR}")
    except Exception as e:
        print(f"[AVISO] No se pudo descargar automáticamente: {e}. Verificar ruta en Drive.")
else:
    print(f"[OK] Modelo LoRA verificado en {MODEL_DIR}")



In [ ]:
import os
from groq import Groq

# Recuperar las claves desde variable de celda, Colab Secrets o entorno.
def _resolver_key(nombre, valor_celda):
    if valor_celda:
        return valor_celda
    try:
        from google.colab import userdata
        v = userdata.get(nombre)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(nombre, "")

api_key = _resolver_key("GROQ_API_KEY", GROQ_API_KEY)
if not api_key:
    raise ValueError(
        "No se encontró la API key de Groq. "
        "Obtén una gratis en https://console.groq.com/keys y pégala en GROQ_API_KEY "
        "o agrégala a Colab Secrets como GROQ_API_KEY."
    )

# Claves adicionales opcionales: suman cupo diario (rotación automática).
API_KEYS = [api_key]
for _sufijo in ["_2", "_3", "_4", "_5"]:
    _extra = _resolver_key(f"GROQ_API_KEY{_sufijo}", globals().get(f"GROQ_API_KEY{_sufijo}", ""))
    if _extra:
        API_KEYS.append(_extra)
API_KEYS = list(dict.fromkeys(API_KEYS))

KEY_IDX = 0
groq_client = Groq(api_key=API_KEYS[0])
print(f"[juez] {len(API_KEYS)} clave(s) de Groq cargada(s).")

def _rotar_clave():
    """Pasa a la siguiente clave con cupo. False si ya no quedan."""
    global KEY_IDX, groq_client
    if KEY_IDX + 1 >= len(API_KEYS):
        return False
    KEY_IDX += 1
    groq_client = Groq(api_key=API_KEYS[KEY_IDX])
    print(f"    [claves] cupo agotado -> rotando a la clave {KEY_IDX + 1}/{len(API_KEYS)}")
    return True

def test_model_ping(model_name: str) -> bool:
    try:
        groq_client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": "hola"}],
            max_tokens=10,
        )
        return True
    except Exception:
        return False

# Verificación directa sin fallback automático para garantizar reproducibilidad exacta
print(f"Verificando disponibilidad de JUDGE_MODEL = '{JUDGE_MODEL}' en Groq...")
if not test_model_ping(JUDGE_MODEL):
    raise RuntimeError(
        f"El modelo juez '{JUDGE_MODEL}' no está disponible en Groq (probablemente deprecado o sin acceso).\n"
        f"Revisar https://console.groq.com/docs/deprecations y actualizar JUDGE_MODEL manualmente.\n"
        f"NO se aplica fallback automático para preservar reproducibilidad entre ejecuciones."
    )
print(f"[OK] Modelo juez configurado '{JUDGE_MODEL}' verificado exitosamente.")

# Comprobar si el modelo soporta response_format={'type': 'json_object'}
SUPPORTS_JSON_MODE = True
try:
    groq_client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": "Responde solo JSON."},
            {"role": "user", "content": 'Genera {"ping": "pong"}'},
        ],
        max_tokens=25,
        response_format={"type": "json_object"},
    )
    print("  Soporte de JSON mode nativo (response_format): SI")
except Exception:
    SUPPORTS_JSON_MODE = False
    print("  Soporte de JSON mode nativo (response_format): NO (se usará parseo robusto por regex)")

print(f"\nJuez LLM listo para evaluación: {JUDGE_MODEL} (Groq API)")



---
## 1. Cargar el Gold Set del equipo

El gold set es un archivo JSONL (`gold_examples.jsonl`) generado de manera centralizada en M1 a partir del split `test`.
Cada ejemplo contiene:
- `input`: texto clínico del documento.
- `esperado`: lista de entidades patológicas anotadas (`list[str]`).
- `criterio`: justificación de selección.

In [8]:
import zipfile
import xml.etree.ElementTree as ET
import ast

def load_gold_from_docx(docx_path: Path) -> list:
    """Extrae los ejemplos gold estructurados desde el archivo Word Ejemplos_Gold.docx."""
    with zipfile.ZipFile(docx_path) as z:
        xml_content = z.read("word/document.xml")
        tree = ET.fromstring(xml_content)
        text = [elem.text for elem in tree.iter("{http://schemas.openxmlformats.org/wordprocessingml/2006/main}t") if elem.text]
        full = "".join(text)
    matches = re.findall(r"(\{[^{}]*?'input'[\s\S]*?\})", full)
    examples = []
    for m in matches:
        s = m.strip()
        if s.endswith(","):
            s = s[:-1]
        s = re.sub(r"(['\"])\s*('esperado')", r"\1, \2", s)
        s = re.sub(r"(['\"])\s*('criterio')", r"\1, \2", s)
        s = re.sub(r"(\])\s*('criterio')", r"\1, \2", s)
        try:
            d = ast.literal_eval(s)
            examples.append(d)
        except Exception:
            pass
    return examples

# Búsqueda inteligente del gold set (JSONL o DOCX en Drive y local)
candidate_paths = [
    GOLD_SET_PATH,
    PROJECT_ROOT / "M2" / "eval_harness" / "gold_examples.jsonl",
    PROJECT_ROOT / "M2" / "gold_examples.jsonl",
    Path("M2/eval_harness/gold_examples.jsonl"),
    Path("M2/gold_examples.jsonl"),
    Path("eval_harness/gold_examples.jsonl"),
    Path("gold_examples.jsonl"),
    Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/eval_harness/gold_examples.jsonl"),
    Path("/content/drive/MyDrive/TopicosIA/M2/eval_harness/gold_examples.jsonl"),
    Path("/content/drive/MyDrive/Topicos-IA/M2/eval_harness/gold_examples.jsonl"),
]

resolved_gold_path = next((p for p in candidate_paths if p.exists()), None)
gold_examples = []

if resolved_gold_path:
    print(f"Cargando gold set desde JSONL: {resolved_gold_path}")
    with open(resolved_gold_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                gold_examples.append(json.loads(line))
else:
    # Buscar Ejemplos_Gold.docx
    candidate_docx = [
        PROJECT_ROOT / "M2" / "Ejemplos_Gold.docx",
        PROJECT_ROOT / "Ejemplos_Gold.docx",
        Path("M2/Ejemplos_Gold.docx"),
        Path("Ejemplos_Gold.docx"),
        Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/Ejemplos_Gold.docx"),
        Path("/content/drive/MyDrive/TopicosIA/M2/Ejemplos_Gold.docx"),
        Path("/content/drive/MyDrive/Topicos-IA/M2/Ejemplos_Gold.docx"),
    ]
    docx_path = next((p for p in candidate_docx if p.exists()), None)
    if docx_path:
        print(f"Extrayendo gold set desde archivo Word de ejemplos: {docx_path}")
        gold_examples = load_gold_from_docx(docx_path)
        # Sincronizar y persistir automáticamente en formato JSONL para el harness
        GOLD_SET_PATH.parent.mkdir(parents=True, exist_ok=True)
        with open(GOLD_SET_PATH, "w", encoding="utf-8") as f:
            for ex in gold_examples:
                f.write(json.dumps(ex, ensure_ascii=False) + "\n")
        print(f"Guardado como JSONL en: {GOLD_SET_PATH}")
    else:
        # Descargar directamente del repositorio GitHub si no está en Drive
        print("[INFO] Buscando gold_examples.jsonl en repositorio GitHub...")
        try:
            import urllib.request
            url = "https://raw.githubusercontent.com/luisNP21/Topicos-IA/Agustin/M2/eval_harness/gold_examples.jsonl"
            GOLD_SET_PATH.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(url, str(GOLD_SET_PATH))
            with open(GOLD_SET_PATH, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        gold_examples.append(json.loads(line))
            print(f"[OK] Descargado exitosamente desde GitHub: {len(gold_examples)} ejemplos.")
        except Exception as e:
            raise FileNotFoundError(
                f"No se encontró gold_examples.jsonl ni Ejemplos_Gold.docx en Drive ni local.\n"
                f"Ubicaciones verificadas: {[str(p) for p in candidate_paths]}\n"
                f"Error al descargar: {e}"
            )

print(f"\nGold set cargado: {len(gold_examples)} ejemplos")
assert len(gold_examples) > 0, "El gold set está vacío"
assert "input" in gold_examples[0], "Falta campo 'input' en el gold set"
assert "esperado" in gold_examples[0], "Falta campo 'esperado' en el gold set"
assert isinstance(gold_examples[0]["esperado"], list), (
    f"'esperado' debe ser list[str], vino como {type(gold_examples[0]['esperado'])}"
)

print("Contrato verificado exitosamente.")
print("Primer ejemplo:")
print("  input[:120]:", gold_examples[0]["input"][:120], "...")
print("  esperado:", gold_examples[0]["esperado"][:5])
print("  n_entidades:", len(gold_examples[0]["esperado"]))

Cargando gold set desde JSONL: /content/drive/MyDrive/TopicosIA/M2/gold_examples.jsonl

Gold set cargado: 59 ejemplos
Contrato verificado exitosamente.
Primer ejemplo:
  input[:120]: Varón soltero de 37 años de edad, trabajador del campo, que es enviado a nuestro Servicio desde otro hospital por no ten ...
  esperado: ['adenopatías', 'adenopatías bilaterales en las cadenas iliacas interna y externa y en las cadenas inguinales', 'adenopatías bilaterales ilíacas e inguinales', 'afectación de las cadenas ganglionares', 'brucelosis']
  n_entidades: 28


---
## 2. Cargar el modelo fine-tuneado (Clinical BERT + LoRA)

Carga del checkpoint base `PlanTL-GOB-ES/roberta-base-biomedical-clinical-es` e inyección de los adaptadores LoRA entrenados en M1.

In [9]:
label_list = ["O", "B-ENFERMEDAD", "I-ENFERMEDAD"]
id2label   = {i: l for i, l in enumerate(label_list)}
label2id   = {l: i for i, l in enumerate(label_list)}

base_model = AutoModelForTokenClassification.from_pretrained(
    BASE_CHECKPOINT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)


# Parche definitivo contra el bug de versión de torchao en Google Colab
import sys
import peft.import_utils
from packaging import version
peft.import_utils.TORCHAO_MINIMUM_VERSION = version.parse("0.0.1")
peft.import_utils.is_torchao_available = lambda: False
for mod in list(sys.modules.values()):
    if hasattr(mod, "TORCHAO_MINIMUM_VERSION"):
        try:
            mod.TORCHAO_MINIMUM_VERSION = version.parse("0.0.1")
        except Exception:
            pass
    if hasattr(mod, "is_torchao_available"):
        try:
            mod.is_torchao_available = lambda: False
        except Exception:
            pass

model = PeftModel.from_pretrained(base_model, str(MODEL_DIR))
model.eval()   # Desactiva dropout para inferencia determinista

if torch.cuda.is_available():
    model = model.to("cuda")

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))

print("Modelo base y adaptador LoRA cargados exitosamente.")
print("Dispositivo:", next(model.parameters()).device)
print("Vocab size:", tokenizer.vocab_size)

# Sanity check con frase médica
frase_prueba = "El paciente presenta diabetes mellitus tipo 2 y antecedentes de hipertension arterial."
tokens_prueba = frase_prueba.split()

inputs_p = tokenizer(
    tokens_prueba, is_split_into_words=True, return_tensors="pt"
).to(model.device)

with torch.no_grad():
    logits_p = model(**inputs_p).logits

pred_ids_p = logits_p.argmax(dim=-1)[0].tolist()
word_ids_p = inputs_p.word_ids(batch_index=0)

print("\nSanity check -- predicciones por palabra:")
seen = set()
for idx, wid in zip(pred_ids_p, word_ids_p):
    if wid is None or wid in seen:
        continue
    print(f"  {tokens_prueba[wid]:<25} -> {id2label[idx]}")
    seen.add(wid)

config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  504MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  504MB            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors: downloading bytes:           |  0.00B            

[transformers] RobertaForTokenClassification LOAD REPORT from: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Modelo base y adaptador LoRA cargados exitosamente.
Dispositivo: cuda:0
Vocab size: 52000

Sanity check -- predicciones por palabra:
  El                        -> O
  paciente                  -> O
  presenta                  -> O
  diabetes                  -> B-ENFERMEDAD
  mellitus                  -> I-ENFERMEDAD
  tipo                      -> I-ENFERMEDAD
  2                         -> I-ENFERMEDAD
  y                         -> O
  antecedentes              -> O
  de                        -> O
  hipertension              -> B-ENFERMEDAD
  arterial.                 -> I-ENFERMEDAD


---
## 3. Funciones de predicción y chunking de M1

Mismas funciones estándar de `harness.ipynb` para segmentación en ventanas con overlap y agregación determinista por documento.

In [10]:
import string

def normalizar_entidad(e):
    """Quita puntuación de borde que es artefacto de tokenizar con .split()."""
    return e.lower().strip().strip(string.punctuation + " ")


def chunk_words(words, window_words, overlap_words):
    """Parte una lista de palabras en ventanas solapadas."""
    if len(words) <= window_words:
        return [(0, words)]

    chunks = []
    step = window_words - overlap_words
    start = 0
    idx = 0
    while start < len(words):
        chunk = words[start : start + window_words]
        chunks.append((idx, chunk))
        if start + window_words >= len(words):
            break
        start += step
        idx += 1
    return chunks


def strip_chunk_suffix(doc_id):
    return re.sub(r"_chunk\d+$", "", doc_id)


def bio_to_entity_set(tokens, tags):
    entities, current = [], []
    for tok, tag in zip(tokens, tags):
        if tag == "B-ENFERMEDAD":
            if current:
                entities.append(" ".join(current))
            current = [tok]
        elif tag == "I-ENFERMEDAD" and current:
            current.append(tok)
        else:
            if current:
                entities.append(" ".join(current))
            current = []
    if current:
        entities.append(" ".join(current))
    return set(normalizar_entidad(e) for e in entities)


def aggregate_entities_by_original_doc(doc_ids, entity_sets):
    grouped = {}
    for doc_id, ents in zip(doc_ids, entity_sets):
        orig_id = strip_chunk_suffix(doc_id)
        grouped.setdefault(orig_id, set()).update(ents)
    return grouped


def micro_prf1_by_doc(true_by_doc, pred_by_doc):
    tp = fp = fn = 0
    all_doc_ids = set(true_by_doc) | set(pred_by_doc)
    for doc_id in all_doc_ids:
        true_ents = true_by_doc.get(doc_id, set())
        pred_ents = pred_by_doc.get(doc_id, set())
        tp += len(true_ents & pred_ents)
        fp += len(pred_ents - true_ents)
        fn += len(true_ents - pred_ents)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"precision": precision, "recall": recall, "f1": f1,
            "tp": tp, "fp": fp, "fn": fn}


def predict_word_level(tokens):
    inputs = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
    ).to(model.device)
    word_ids = inputs.word_ids(batch_index=0)

    with torch.no_grad():
        logits = model(**inputs).logits
    pred_ids = logits.argmax(dim=-1)[0].tolist()

    word_preds, seen = [], set()
    for idx, wid in zip(pred_ids, word_ids):
        if wid is None or wid in seen:
            continue
        word_preds.append(id2label[idx])
        seen.add(wid)
    return word_preds


def predict_entities_chunked(text, window_words, overlap_words):
    words = text.split()
    chunks = chunk_words(words, window_words, overlap_words)

    resultados = []
    for chunk_idx, chunk_words_list in chunks:
        pred_tags = predict_word_level(chunk_words_list)
        entity_set = bio_to_entity_set(chunk_words_list, pred_tags)
        suffix = f"_chunk{chunk_idx}" if len(chunks) > 1 else ""
        resultados.append((suffix, entity_set))
    return resultados


def dimension_metrica_clasica(gold_examples, window_words, overlap_words):
    true_by_doc = {}
    pred_doc_ids, pred_entity_sets = [], []

    for i, ex in enumerate(gold_examples):
        doc_id = f"ex_{i}"
        true_by_doc[doc_id] = {normalizar_entidad(e) for e in ex["esperado"]}

        resultados_chunks = predict_entities_chunked(ex["input"], window_words, overlap_words)
        for suffix, entity_set in resultados_chunks:
            pred_doc_ids.append(doc_id + suffix)
            pred_entity_sets.append(entity_set)

    pred_by_doc = aggregate_entities_by_original_doc(pred_doc_ids, pred_entity_sets)
    metrics = micro_prf1_by_doc(true_by_doc, pred_by_doc)
    return metrics, true_by_doc, pred_by_doc

print("Funciones de inferencia y evaluación cargadas.")

Funciones de inferencia y evaluación cargadas.


---
## 4. Formato y preparación de Rich Examples para el Juez

Cargamos los ejemplos directamente desde `resultado_dimension1.json` (`DIM1_PATH`), garantizando que:
1. Usamos exactamente los mismos ejemplos del gold set que la Dimensión 1 y la Dimensión 2 (sin discrepancias de subset).
2. Si `DIM1_PATH` ya existe, se lee instantáneamente; si no existe, se calcula ejecutando `dimension_metrica_clasica` sobre `gold_examples`.

In [ ]:
if DIM1_PATH.exists():
    print(f"Cargando predicciones existentes desde: {DIM1_PATH}")
    with open(DIM1_PATH, "r", encoding="utf-8") as f:
        dim1 = json.load(f)
    true_by_doc_dim1 = dim1["true_by_doc"]
    pred_by_doc_dim1 = dim1["pred_by_doc"]
else:
    print(f"No se encontró {DIM1_PATH}. Evaluando gold set con el modelo fine-tuneado (chunking, window={WINDOW_WORDS})...")
    metrics_exact, true_by_doc, pred_by_doc = dimension_metrica_clasica(
        gold_examples, WINDOW_WORDS, OVERLAP_WORDS
    )
    true_by_doc_dim1 = {k: sorted(v) for k, v in true_by_doc.items()}
    pred_by_doc_dim1 = {k: sorted(v) for k, v in pred_by_doc.items()}
    resultado_dim1 = {
        "dimension": "metrica_clasica_exact_match",
        "modelo": BASE_CHECKPOINT,
        "adaptador_lora": str(MODEL_DIR),
        "gold_set": str(GOLD_SET_PATH),
        "n_ejemplos_evaluados": len(true_by_doc),
        "metrics": metrics_exact,
        "true_by_doc": true_by_doc_dim1,
        "pred_by_doc": pred_by_doc_dim1,
    }
    with open(DIM1_PATH, "w", encoding="utf-8") as f:
        json.dump(resultado_dim1, f, indent=2, ensure_ascii=False)
    print(f"Predicciones evaluadas y guardadas en {DIM1_PATH}")

# El doc_id de la Dimension 1 es 'ex_{i}', con i = indice del ejemplo en el
# gold set. Por eso se puede recuperar el `criterio` de cada caso.
rich_examples = []
for doc_id in sorted(true_by_doc_dim1, key=lambda d: int(d.split("_")[1])):
    idx = int(doc_id.split("_")[1])
    gold = sorted(true_by_doc_dim1[doc_id])
    pred = sorted(pred_by_doc_dim1.get(doc_id, []))
    ex = gold_examples[idx] if idx < len(gold_examples) else {}
    rich_examples.append({
        "doc_id": doc_id,
        "n_gold": len(gold),
        "gold": gold,
        "pred": pred,
        "criterio": ex.get("criterio", ""),
        "input": ex.get("input", ""),
    })

print(f"Ejemplos cargados para el juez: {len(rich_examples)}")
print("Cada ejemplo incluye el `criterio` del gold set (Dimensión de dominio de S06).")

for doc in rich_examples[:3]:
    print(f"\ndoc_id: {doc['doc_id']} | n_gold: {doc['n_gold']}")
    print(f"  Gold: {doc['gold']}")
    print(f"  Pred: {doc['pred']}")
    print(f"  Criterio[:160]: {doc['criterio'][:160]}")



---
## 5. RÚBRICA del juez LLM (S06)

El juez puntúa 4 dimensiones clínicas (1–5 cada una) con **anclas explícitas**, tal como enseña S06:

| Dimensión | Descripción |
|---|---|
| **Completitud** | ¿Capturó la mayoría de las enfermedades mencionadas? |
| **Exactitud de boundary** | ¿Los nombres coinciden con el gold (o casi)? |
| **Relevancia clínica** | ¿Las predicciones son términos clínicamente válidos? |
| **Ausencia de ruido** | ¿Evitó marcar términos que NO son enfermedades? |

Además el juez responde la **pregunta binaria de dominio** `cumple_criterio` ("si"/"no") leyendo el campo `criterio` de cada caso. `score_juez` = promedio de las 4 dimensiones; `cumple_criterio` alimenta la **dimensión sí/no de dominio** (S06, Dimensión 3).



In [ ]:
RUBRICA_SYSTEM = (
    "Eres un médico especialista en terminología clínica y anotación de entidades. "
    "Evalúas la calidad de una lista de entidades ENFERMEDAD detectadas por un sistema de NER "
    "sobre texto clínico en español. Evalúa SOLO el contenido, con criterio estricto y objetivo. "
    "No sabes qué modelo generó la predicción."
)

RUBRICA_TEMPLATE = """# Evaluación de NER clínico — RÚBRICA (S06)

## Criterio de evaluación de ESTE caso
{criterio}

## Referencia (gold standard)
Enfermedades correctas: {gold_str}

## Predicción del sistema
Enfermedades detectadas: {pred_str}

## Instrucciones
Evalúa la predicción del 1 al 5 en cada dimensión, con estas anclas explícitas:

1. **Completitud** (¿capturó la mayoría de las enfermedades gold?):
   5 = todas o casi todas | 3 = aproximadamente la mitad | 1 = prácticamente ninguna
2. **Exactitud de boundary** (¿los nombres coinciden con el gold?):
   5 = coincidencia exacta o diferencia mínima | 3 = algunos coinciden, otros truncados | 1 = no se parecen
3. **Relevancia clínica** (¿las predicciones son términos de enfermedad válidos?):
   5 = todas válidas | 3 = mezcla | 1 = mayoría inválidas
4. **Ausencia de ruido** (¿evitó marcar términos que NO son enfermedades?):
   5 = sin FP notables | 3 = algunos FP | 1 = demasiados FP

Además responde la pregunta binaria de dominio, que es la que define el
"acierto de dominio" del harness (S06, Dimensión 3):

- **cumple_criterio**: responde "si" cuando la predicción es ÚTIL para el
  criterio de ESTE caso; "no" cuando no lo es. Regla objetiva para decidir:
    * "si" si la mayoría de las enfermedades gold están capturadas (completitud
      >= 4) Y las que se detectan son clínicamente válidas (relevancia >= 4) Y no
      hay ruido grave (ausencia_ruido >= 3). Los errores leves de boundary/formato
      NO impiden el "si": una lista útil con un límite de span imperfecto sigue
      sirviendo a un clínico.
    * "no" si falta una parte sustancial de las enfermedades, o si domina el
      ruido (falsos positivos), o si hay entidades inventadas.
  Referencia cuantitativa (úsala para no dudar): si la predicción captura
  aproximadamente el 75% o más de las enfermedades gold y no tiene ruido grave,
  la respuesta es "si" aunque falte alguna entidad. Ejemplo: gold de 4 entidades
  y la predicción acierta 3 -> es "si" (cumple). gold de 4 y acierta 2 -> es "no".
  No respondas "si" por cortesía, pero tampoco exijas perfección: el "si" es
  "esto sirve para el criterio del caso", no "esto es idéntico al gold".

## Respuesta
Responde ÚNICAMENTE con JSON válido:
```json
{{
  "completitud": <1-5>,
  "exactitud_boundary": <1-5>,
  "relevancia_clinica": <1-5>,
  "ausencia_ruido": <1-5>,
  "cumple_criterio": "si" | "no",
  "justificacion": "<una oración breve>"
}}
```"""

PAIRWISE_SYSTEM = (
    "Eres un evaluador estricto y objetivo de anotación de entidades clínicas (NER). "
    "Comparas dos listas anónimas de entidades ENFERMEDAD y eliges la mejor. "
    "No conoces el origen de ninguna de las dos."
)

PAIRWISE_TEMPLATE = """# Comparación ciega de dos anotaciones de NER clínico

## Criterio del caso
{criterio}

## Candidato A
{resp_a}

## Candidato B
{resp_b}

## Pregunta
¿Cuál lista de entidades ENFERMEDAD es mejor: la A o la B?
Responde SOLO con la letra A o B (una sola letra, sin explicación)."""

DIM_KEYS = ["completitud", "exactitud_boundary", "relevancia_clinica", "ausencia_ruido"]
CRITERIO_DEFECTO = ("La respuesta debe identificar todas las menciones de enfermedad presentes "
                    "en el texto clínico, sin incluir términos que no sean enfermedades ni menciones alucinadas.")


def build_judge_prompt(gold, pred, criterio=""):
    """Prompt pointwise. Ya NO existe 'order': no se intercambian gold/pred."""
    gold_str = ", ".join(gold) if gold else "(ninguna)"
    pred_str = ", ".join(pred) if pred else "(ninguna)"
    return RUBRICA_TEMPLATE.format(criterio=(criterio or "").strip() or CRITERIO_DEFECTO,
                                   gold_str=gold_str, pred_str=pred_str)


def build_pairwise_prompt(criterio, resp_a, resp_b):
    a = ", ".join(resp_a) if resp_a else "(ninguna)"
    b = ", ".join(resp_b) if resp_b else "(ninguna)"
    return PAIRWISE_TEMPLATE.format(criterio=(criterio or "").strip() or CRITERIO_DEFECTO,
                                    resp_a=a, resp_b=b)


def parse_judge_response(text):
    for pattern in [r"```(?:json)?\s*({[\s\S]*?})\s*```", r"({[\s\S]*?})"]:
        m = re.search(pattern, text)
        if m:
            try:
                return json.loads(m.group(1))
            except Exception:
                pass
    try:
        return json.loads(text.strip())
    except Exception:
        pass
    result = {}
    for key in DIM_KEYS:
        val_m = re.search(rf'"{key}"\s*:\s*([1-5])', text, re.IGNORECASE)
        result[key] = int(val_m.group(1)) if val_m else None
    cc_m = re.search(r'"cumple_criterio"\s*:\s*"?\s*(si|sí|no|true|false)', text, re.IGNORECASE)
    result["cumple_criterio"] = cc_m.group(1) if cc_m else None
    just_m = re.search(r'"justificacion"\s*:\s*"([^"\n]+)', text, re.IGNORECASE)
    result["justificacion"] = just_m.group(1) if just_m else (text[-200:].strip() if text else "Sin texto")
    return result


def parse_pairwise_response(text):
    """Letra A/B del veredicto. Prioriza una letra independiente para no
    confundir la 'A' de 'LA'/'MEJOR' con un veredicto. '?' si no hay letra."""
    upper = text.upper()
    standalone = re.search(r"(?<![A-Z])([AB])(?![A-Z])", upper)
    if standalone:
        return standalone.group(1)
    m = re.search(r"[AB]", upper)
    return m.group() if m else "?"


def normalizar_cumple_criterio(val):
    if val is None:
        return None
    if isinstance(val, bool):
        return val
    s = str(val).strip().lower()
    if s in ("si", "sí", "s", "true", "yes", "1"):
        return True
    if s in ("no", "n", "false", "0"):
        return False
    return None


def compute_score(d):
    vals = [d.get(k) for k in DIM_KEYS if d.get(k) is not None]
    return float(np.mean(vals)) if vals else None


def _llm_complete(system, user, retry=5, sleep_s=1.0, use_json=False):
    """Llamada base con reintentos, espaciado y rotación de claves ante cupo agotado."""
    global JUDGE_MAX_TOKENS
    use_json_mode = use_json and globals().get("SUPPORTS_JSON_MODE", True)
    rate_limit_seguidos = 0

    for attempt in range(retry):
        try:
            # Espaciado preventivo contra los límites por minuto de Groq.
            time.sleep(JUDGE_PAUSA_ENTRE_LLAMADAS)

            kwargs = {
                "model": JUDGE_MODEL,
                "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
                "temperature": JUDGE_TEMPERATURE,
                "max_tokens": JUDGE_MAX_TOKENS,
            }
            if use_json_mode:
                kwargs["response_format"] = {"type": "json_object"}

            response = groq_client.chat.completions.create(**kwargs)
            content = response.choices[0].message.content or ""
            finish = getattr(response.choices[0], "finish_reason", None)

            # Modelos de razonamiento (p.ej. gpt-oss) consumen tokens internos: si
            # finish_reason == 'length' el presupuesto se agotó antes del JSON.
            if finish == "length" and not content.strip():
                nuevo_limite = min(JUDGE_MAX_TOKENS * 2, 8000)
                if nuevo_limite > kwargs["max_tokens"]:
                    print(f"    [Ajuste] finish_reason='length' sin contenido -> "
                          f"max_tokens {kwargs['max_tokens']} -> {nuevo_limite}")
                    JUDGE_MAX_TOKENS = nuevo_limite
                    continue

            time.sleep(sleep_s)
            return content

        except Exception as e:
            err_str = str(e)
            if "response_format" in err_str or "json_object" in err_str:
                use_json_mode = False
            is_rate_limit = "429" in err_str or "rate_limit" in err_str.lower()

            if is_rate_limit:
                # Cupo DIARIO agotado (TPD): no se resuelve esperando, se rota la clave.
                cupo_diario = "tokens per day" in err_str or "(TPD)" in err_str
                if cupo_diario and _rotar_clave():
                    rate_limit_seguidos = 0
                    continue

                rate_limit_seguidos += 1
                espera = JUDGE_PAUSA_ENTRE_LLAMADAS
                m = re.search(r"try again in ([0-9.]+)s", err_str)
                if m:
                    espera = max(espera, float(m.group(1)) + 2)
                else:
                    espera = max(espera, 10 * rate_limit_seguidos)
                print(f"    [429] límite de tasa; esperando {espera:.1f}s...")
                time.sleep(espera)
                if rate_limit_seguidos >= retry:
                    retry = min(retry + 8, 40)
                continue

            print(f"    [Error del juez] {type(e).__name__}: {err_str[:300]}")
            wait_time = 2 * (attempt + 1)
            print(f"    [Reintento {attempt + 1}/{retry}] esperando {wait_time}s...")
            time.sleep(wait_time)
    return ""


def call_judge(gold, pred, criterio="", retry=5):
    """Juez pointwise: 4 dimensiones 1-5 + veredicto binario de dominio."""
    prompt = build_judge_prompt(gold, pred, criterio)
    content = _llm_complete(RUBRICA_SYSTEM, prompt, retry=retry, use_json=True)
    parsed = parse_judge_response(content)
    if not any(parsed.get(k) is not None for k in DIM_KEYS):
        content = _llm_complete(RUBRICA_SYSTEM, prompt, retry=retry, use_json=False)
        parsed = parse_judge_response(content)
    if any(parsed.get(k) is not None for k in DIM_KEYS):
        return parsed
    return {**{k: None for k in DIM_KEYS}, "cumple_criterio": None, "justificacion": "JUDGE_CALL_FAILED"}


def call_judge_pairwise(criterio, resp_a, resp_b, retry=5):
    """Comparación ciega A/B (S06, Lab B)."""
    prompt = build_pairwise_prompt(criterio, resp_a, resp_b)
    return parse_pairwise_response(_llm_complete(PAIRWISE_SYSTEM, prompt, retry=retry, use_json=False))


test_res = call_judge(["asma crónica"], ["asma"], "La respuesta debe identificar las enfermedades del texto.")
print("Test call_judge verificado:", test_res)



---
## 6. Sesgo de posición — protocolo pairwise A/B (S06, Lab B)

### Qué es el sesgo de posición
Un juez LLM tiende a preferir la respuesta que ve primero (o siempre la "A"), sin importar el contenido.

### Protocolo correcto (S06)
1. Se presentan **dos respuestas anónimas A/B** al juez y se le pide elegir la mejor.
2. Se repite la comparación **intercambiando el orden**.
3. Solo se declara ganador si el veredicto **coincide en ambos órdenes**; si el juez se contradice → **empate** (el par no es confiable).

### Qué NO se hace
- ❌ Intercambiar `gold` y `pred` dentro de la rúbrica pointwise: eso mide la asimetría de la rúbrica, no el orden.
- ❌ Promediar el score "normal" y el "invertido" para el score final: eso contamina la métrica.

El test de posición es un **diagnóstico**: el score final del juez sigue siendo el pointwise.



In [ ]:
print(f"Corriendo juez pointwise + test de posición sobre {len(rich_examples)} rich examples...")
print(f"  - Pointwise: 1 llamada por documento")
print(f"  - Posición:  2 llamadas pairwise por documento (A/B + B/A) cuando pred != gold\n")

position_results = []

for i, doc in enumerate(rich_examples):
    print(f"  [{i+1}/{len(rich_examples)}] {doc['doc_id']} | n_gold={doc['n_gold']} | n_pred={len(doc['pred'])}")

    # 1+2: score final pointwise (una sola llamada) y veredicto de dominio
    result_pointwise = call_judge(doc["gold"], doc["pred"], doc["criterio"])
    score_punto = compute_score(result_pointwise)
    cumple = normalizar_cumple_criterio(result_pointwise.get("cumple_criterio"))

    # 3: sesgo de posición pairwise A/B (solo si pred != gold)
    veredicto = None
    v_pred_a = v_gold_a = None
    if set(doc["pred"]) != set(doc["gold"]):
        v_pred_a = call_judge_pairwise(doc["criterio"], doc["pred"], doc["gold"])  # pred=A, gold=B
        v_gold_a = call_judge_pairwise(doc["criterio"], doc["gold"], doc["pred"])  # gold=A, pred=B
        if v_pred_a == "B" and v_gold_a == "A":
            veredicto = "gold"
        elif v_pred_a == "A" and v_gold_a == "B":
            veredicto = "pred"
        else:
            veredicto = "empate"

    s_str = f"{score_punto:.2f}" if score_punto is not None else "FAIL"
    print(f"      score pointwise: {s_str} | cumple_criterio: {cumple} | posición: {veredicto} (A/B={v_pred_a}/{v_gold_a})")

    position_results.append({
        "doc_id": doc["doc_id"],
        "n_gold": doc["n_gold"],
        "n_pred": len(doc["pred"]),
        "gold": doc["gold"],
        "pred": doc["pred"],
        "criterio": doc["criterio"],
        "score_juez": score_punto,
        "cumple_criterio": cumple,
        "veredicto_posicion": veredicto,
        "v_pred_primero": v_pred_a,
        "v_gold_primero": v_gold_a,
        "result_pointwise": result_pointwise,
    })

print("\nListo.")



In [ ]:
df_pos = pd.DataFrame(position_results)

# --- Score final (pointwise) ---
scores = df_pos["score_juez"].dropna()
print("=" * 60)
print("ANÁLISIS — JUEZ POINTWISE (score final)")
print("=" * 60)
print(f"  Documentos evaluados         : {len(df_pos)}")
print(f"  Score juez medio (pointwise) : {scores.mean():.3f} ± {scores.std():.3f}")
print()

# --- Dimensión de dominio (sí/no) ---
validos = df_pos["cumple_criterio"].dropna()
print("=" * 60)
print("ANÁLISIS — DIMENSIÓN DE DOMINIO (S06, Dimensión 3)")
print("=" * 60)
if len(validos):
    aciertos = int(validos.sum())
    print(f"  Aciertos (cumple_criterio = si): {aciertos}/{len(validos)} ({aciertos/len(validos):.1%})")
else:
    print("  Sin datos")
print()

# --- Sesgo de posición (pairwise A/B) ---
veredictos = df_pos["veredicto_posicion"].dropna()
n_pares = len(veredictos)
n_gold = int((veredictos == "gold").sum())
n_pred = int((veredictos == "pred").sum())
n_emp = int((veredictos == "empate").sum())
tasa = n_emp / n_pares if n_pares else 0.0
print("=" * 60)
print("ANÁLISIS — SESGO DE POSICIÓN (pairwise A/B, S06 Lab B)")
print("=" * 60)
print(f"  Pares comparables (pred != gold): {n_pares}")
print(f"  Gana la referencia (gold) en ambos órdenes : {n_gold}")
print(f"  Gana la predicción (pred) en ambos órdenes : {n_pred}")
print(f"  Empates (el juez se contradijo por el orden): {n_emp}")
print(f"  Tasa de sesgo de posición: {tasa:.1%}")
if n_pares == 0:
    print("  Sin pares comparables.")
elif tasa <= 0.2:
    print("  Veredicto: estable respecto al orden.")
elif tasa <= 0.5:
    print("  Veredicto: sesgo moderado — reportar los empates como no confiables.")
else:
    print("  Veredicto: sesgo fuerte — el veredicto depende del orden.")
print()
print("Resumen por documento:")
print(df_pos[["doc_id", "n_gold", "n_pred", "score_juez", "cumple_criterio", "veredicto_posicion"]].to_string())



---
## 7. Sesgo de longitud

### Qué es el sesgo de longitud
El juez puede puntuar más alto respuestas más largas, independientemente de si son correctas.

### Protocolo de detección (corregido)
Pares de control con asimetría de longitud, donde la respuesta "larga" es **parcialmente correcta** (algunos aciertos + ruido/FP), no ruido sin relación con el gold:
- **Tipo A** — corta **correcta y precisa** vs. larga **parcialmente correcta**.
- **Tipo B** — larga **correcta y completa** vs. corta **incompleta (recall parcial)**.

En ambos casos el juez debería premiar la respuesta de mayor calidad, no la más larga.



In [ ]:
LENGTH_BIAS_PAIRS = [
    # Tipo A — correcta corta y precisa vs. larga PARCIALMENTE correcta (algunos aciertos + FP)
    {
        "tipo": "A", "descripcion": "Correcta corta y precisa vs. larga parcialmente correcta (pocos aciertos + FP)",
        "criterio": "La respuesta debe identificar las menciones de enfermedad del texto sin agregar términos que no sean enfermedades ni menciones alucinadas.",
        "gold": ["neumonía", "sepsis"],
        "pred_correcta": ["neumonía", "sepsis"],
        "pred_incorrecta": ["neumonía", "dislipidemia", "hipotiroidismo", "artritis reumatoide",
                             "fibromialgia", "esclerosis múltiple", "enfermedad de crohn", "psoriasis"],
    },
    {
        "tipo": "A", "descripcion": "Correcta corta y precisa vs. larga parcialmente correcta (pocos aciertos + FP)",
        "criterio": "La respuesta debe identificar las menciones de enfermedad del texto sin agregar términos que no sean enfermedades ni menciones alucinadas.",
        "gold": ["infarto de miocardio"],
        "pred_correcta": ["infarto de miocardio"],
        "pred_incorrecta": ["infarto de miocardio", "gastritis crónica", "úlcera péptica",
                             "reflujo gastroesofágico", "colelitiasis", "pancreatitis aguda",
                             "apendicitis", "diverticulitis", "colitis ulcerosa"],
    },
    {
        "tipo": "A", "descripcion": "Correcta corta y precisa vs. larga parcialmente correcta (pocos aciertos + FP)",
        "criterio": "La respuesta debe identificar las menciones de enfermedad del texto sin agregar términos que no sean enfermedades ni menciones alucinadas.",
        "gold": ["carcinoma de células renales"],
        "pred_correcta": ["carcinoma de células renales"],
        "pred_incorrecta": ["carcinoma de células renales", "cefalea tensional", "migraña",
                             "vértigo posicional", "neuralgia del trigémino", "parálisis facial"],
    },
    # Tipo B — correcta larga y completa vs. incorrecta corta (recall parcial)
    {
        "tipo": "B", "descripcion": "Correcta larga y completa vs. incorrecta corta (recall parcial)",
        "criterio": "La respuesta debe identificar las menciones de enfermedad del texto sin agregar términos que no sean enfermedades ni menciones alucinadas.",
        "gold": ["diabetes mellitus tipo 2", "hipertensión arterial", "dislipidemia", "obesidad"],
        "pred_correcta": ["diabetes mellitus tipo 2", "hipertensión arterial", "dislipidemia", "obesidad"],
        "pred_incorrecta": ["diabetes mellitus tipo 2"],
    },
    {
        "tipo": "B", "descripcion": "Correcta larga y completa vs. incorrecta corta (recall parcial)",
        "criterio": "La respuesta debe identificar las menciones de enfermedad del texto sin agregar términos que no sean enfermedades ni menciones alucinadas.",
        "gold": ["neumonía adquirida en la comunidad", "derrame pleural", "insuficiencia respiratoria"],
        "pred_correcta": ["neumonía adquirida en la comunidad", "derrame pleural", "insuficiencia respiratoria"],
        "pred_incorrecta": ["neumonía adquirida en la comunidad"],
    },
    {
        "tipo": "B", "descripcion": "Correcta larga y completa vs. incorrecta corta (recall parcial)",
        "criterio": "La respuesta debe identificar las menciones de enfermedad del texto sin agregar términos que no sean enfermedades ni menciones alucinadas.",
        "gold": ["carcinoma ductal infiltrante", "metástasis hepática", "anemia", "trombocitopenia", "neutropenia febril"],
        "pred_correcta": ["carcinoma ductal infiltrante", "metástasis hepática", "anemia", "trombocitopenia", "neutropenia febril"],
        "pred_incorrecta": ["carcinoma ductal infiltrante"],
    },
]

print(f"Pares de control de longitud: {len(LENGTH_BIAS_PAIRS)} pares definidos.")



In [ ]:
print("Corriendo juez sobre pares de longitud...")
length_results = []

for i, pair in enumerate(LENGTH_BIAS_PAIRS):
    print(f"  [{i+1}/{len(LENGTH_BIAS_PAIRS)}] Tipo {pair['tipo']} — {pair['descripcion']}")
    criterio = pair.get("criterio", "")
    r_cor = call_judge(pair["gold"], pair["pred_correcta"], criterio)
    r_inc = call_judge(pair["gold"], pair["pred_incorrecta"], criterio)
    sc, si = compute_score(r_cor), compute_score(r_inc)
    length_results.append({
        "tipo": pair["tipo"],
        "descripcion": pair["descripcion"],
        "n_correcta": len(pair["pred_correcta"]),
        "n_incorrecta": len(pair["pred_incorrecta"]),
        "score_correcta": sc,
        "score_incorrecta": si,
        "juez_premia_calidad": (sc > si) if (sc is not None and si is not None) else None,
    })
    sc_str = f"{sc:.2f}" if sc is not None else "FAIL"
    si_str = f"{si:.2f}" if si is not None else "FAIL"
    ok_str = "OK" if (sc is not None and si is not None and sc > si) else "SESGO"
    print(f"      Score correcta: {sc_str} | Score incorrecta: {si_str} -> {ok_str}")

print("\nListo.")



In [17]:
df_len  = pd.DataFrame(length_results)
n_ok    = df_len["juez_premia_calidad"].sum()
n_total = df_len["juez_premia_calidad"].notna().sum()

print("=" * 55)
print("ANÁLISIS — SESGO DE LONGITUD")
print("=" * 55)
print(f"  Pares evaluados: {n_total}")
print(f"  El juez premió calidad sobre longitud: {n_ok}/{n_total} ({n_ok/n_total*100:.0f}%)")
print()

if n_ok / n_total >= 0.8:
    print("El juez NO muestra sesgo de longitud relevante.")
elif n_ok / n_total >= 0.6:
    print("Sesgo LEVE de longitud — en algunos casos premia extensión.")
else:
    print("Sesgo FUERTE de longitud — agregar instrucción explícita en la rúbrica.")

print()
print(df_len[["tipo", "n_correcta", "n_incorrecta", "score_correcta", "score_incorrecta", "juez_premia_calidad"]])

ANÁLISIS — SESGO DE LONGITUD
  Pares evaluados: 5
  El juez premió calidad sobre longitud: 5/5 (100%)

El juez NO muestra sesgo de longitud relevante.

  tipo  n_correcta  n_incorrecta  score_correcta  score_incorrecta  \
0    A           1             8            4.50               3.0   
1    A           1             9            4.00               3.0   
2    A           1             5            4.50               3.0   
3    B           3             1            4.75               3.0   
4    B           2             1            4.75               1.0   

   juez_premia_calidad  
0                 True  
1                 True  
2                 True  
3                 True  
4                 True  


---
## 8. Sesgo de auto-preferencia (documentación)

### Qué es
Un LLM juez tiende a preferir outputs de modelos de su misma familia arquitectural o de entrenamiento.

### Situación en este proyecto
- **Juez:** Modelo externo vía Groq (`openai/gpt-oss-120b`).
- **Modelo evaluado:** `roberta-base-biomedical-clinical-es` (encoder RoBERTa, PlanTL / BSC).
- **Modelo alternativo del proyecto:** `mT5` (encoder-decoder, Google).

Al usar un juez LLM externo vía Groq, el juez **no pertenece a la familia de ninguno de los modelos del proyecto**, lo cual mitiga el sesgo de auto-preferencia por diseño.

### Protocolo de mitigación activo
1. **Anonimización total:** El prompt del juez nunca menciona el modelo que generó las predicciones.
2. **Formato idéntico:** Tanto gold como predicciones se presentan como listas de strings.

In [18]:
sample_prompt = build_judge_prompt(["neumonía", "sepsis"], ["neumonía"])
forbidden     = ["roberta", "bert", "mt5", "llama", "gpt", "gemini", "plantl", "biomedical", "clinical"]
found = [n for n in forbidden if n.lower() in sample_prompt.lower()]

if not found:
    print("El prompt NO menciona ningún modelo — output completamente anonimizado.")
else:
    print(f"ATENCIÓN: el prompt menciona {found} — revisar RUBRICA_TEMPLATE.")

print()
print("Resumen de mitigaciones de auto-preferencia:")
print("  1. Output anonimizado (sin nombres de modelo en el prompt): OK")
print("  2. Formato estandarizado (listas de strings para todos los modelos): OK")
print("  3. Juez externo a las familias evaluadas (Groq LLM vs RoBERTa/mT5): OK")

El prompt NO menciona ningún modelo — output completamente anonimizado.

Resumen de mitigaciones de auto-preferencia:
  1. Output anonimizado (sin nombres de modelo en el prompt): OK
  2. Formato estandarizado (listas de strings para todos los modelos): OK
  3. Juez externo a las familias evaluadas (Groq LLM vs RoBERTa/mT5): OK


---
## 9. Scorecard final sobre rich examples

Usamos el **score pointwise** (Sección 6) como score final del juez, y agregamos la **dimensión de dominio** (`cumple_criterio`). El test de posición es solo diagnóstico y no modifica el score.



In [ ]:
def compute_exact_f1_doc(gold, pred):
    """F1 exact-match para un documento (para triangular con la métrica clásica)."""
    g, p = set(gold), set(pred)
    tp, fp, fn = len(g & p), len(p - g), len(g - p)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return {"precision": prec, "recall": rec, "f1": f1}


scorecard_rows = []
for row in position_results:
    exact = compute_exact_f1_doc(row["gold"], row["pred"])
    scorecard_rows.append({
        "doc_id": row["doc_id"],
        "n_gold": row["n_gold"],
        "n_pred": row["n_pred"],
        "f1_exacto": round(exact["f1"], 3),
        "prec_exacta": round(exact["precision"], 3),
        "rec_exacto": round(exact["recall"], 3),
        "score_juez": round(row["score_juez"], 3) if row["score_juez"] is not None else None,
        "cumple_criterio": row["cumple_criterio"],
        "veredicto_posicion": row["veredicto_posicion"],
        "gold": row["gold"],
        "pred": row["pred"],
    })

df_sc = pd.DataFrame(scorecard_rows)

print("=" * 64)
print("SCORECARD FINAL — JUEZ POINTWISE vs. F1 EXACTO")
print("=" * 64)
print(f"Documentos evaluados : {len(df_sc)}")
print(f"F1 exacto medio      : {df_sc['f1_exacto'].mean():.3f}")
print(f"Score juez medio     : {df_sc['score_juez'].mean():.3f} / 5.0")
_validos = df_sc["cumple_criterio"].dropna()
print(f"Aciertos de dominio  : {int(_validos.sum())}/{len(_validos)}")
print(f"Correlación Pearson  : {df_sc['f1_exacto'].corr(df_sc['score_juez']):.3f}")
print()
print(df_sc[["doc_id", "n_gold", "n_pred", "f1_exacto", "score_juez", "cumple_criterio", "veredicto_posicion"]].to_string())



---
## 10. Interpretación cualitativa — ¿Qué revela el juez que el F1 exacto no capta?

| Patrón | F1 exacto | Score juez | Interpretación |
|--------|-----------|-----------|----------------|
| **A** | ≈ 0 | ≥ 3.5 | Entendió la entidad pero con **boundary distinto** — error de span, no de comprensión. |
| **B** | > 0.3 | ≤ 2.5 | Acertó la string exacta pero el output **no es clínicamente satisfactorio**. |
| **C** | ≈ 0 | ≤ 2.0 | Fallo completo. |
| **D** | ≥ 0.7 | ≥ 4.0 | Funcionamiento correcto — referencia positiva. |
| **E** | resto | resto | Caso mixto. |

In [20]:
def classify_pattern(row):
    s, f = row["score_juez"], row["f1_exacto"]
    if s is None or f is None: return "? — datos faltantes"
    if f < 0.1 and s >= 3.5:   return "A — boundary error (comprensión OK, span malo)"
    if f > 0.3 and s <= 2.5:   return "B — string OK, calidad clínica baja"
    if f < 0.1 and s <= 2.0:   return "C — fallo completo"
    if f >= 0.7 and s >= 4.0:  return "D — referencia positiva"
    return "E — caso mixto"


df_sc["patron"] = df_sc.apply(classify_pattern, axis=1)
print("Distribución de patrones:")
print(df_sc["patron"].value_counts().to_string())

for patron_id in ["A", "B", "C", "D"]:
    subset = df_sc[df_sc["patron"].str.startswith(patron_id)]
    if not subset.empty:
        ej = subset.iloc[0]
        print(f"\nEjemplo Patrón {patron_id} ({ej['doc_id']}):")
        print(f"  F1 exacto: {ej['f1_exacto']} | Score juez: {ej['score_juez']}")
        print(f"  Gold     : {ej['gold']}")
        print(f"  Pred     : {ej['pred']}")

Distribución de patrones:
patron
E — caso mixto             35
D — referencia positiva    24

Ejemplo Patrón D (ex_12):
  F1 exacto: 0.789 | Score juez: 4.125
  Gold     : ['ascitis, secundaria a peritonitis por echerichia coli', 'calcificación peritoneal', 'esclerosis y calcificación de toda la membrana peritoneal', 'eventración peritoneal en el orificio del catéter', 'hiperparatiroidismo grave', 'hipogonadismo', 'infección de túnel por el estafilococo', 'insuficiencia renal', 'isquemia aguda en miembro inferior', 'obesidad', 'obstrucción femoropoplítea', 'peritonitis', 'peritonitis por echerichia coli', 'peritonitis, causada por dos gérmenes (staphilococcus epidermidis y pseudomona aeruginosa', 'polidactilia', 'retinitis pigmentaria', 'retraso mental', 'síndrome de laurence-moon-biell', 'trombosis venosa']
  Pred     : ['ascitis', 'calcificación peritoneal', 'esclerosis y calcificación de toda la membrana peritoneal', 'eventración peritoneal en el orificio del catéter', 'hiperparatir

In [ ]:
import json as _j

OUTPUT_DIR = PROJECT_ROOT / "M2" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUTPUT_DIR / "llm_judge_scorecard.csv"
df_sc.drop(columns=["gold", "pred"], errors="ignore").to_csv(csv_path, index=False, encoding="utf-8")
print(f"Scorecard  -> {csv_path}")

_scores = df_sc["score_juez"].dropna()
_validos = df_sc["cumple_criterio"].dropna()
_vered = df_sc["veredicto_posicion"].dropna()
_n_pares = len(_vered)
_n_emp = int((_vered == "empate").sum())

bias_summary = {
    "judge_model": JUDGE_MODEL,
    "judge_provider": "Groq",
    "base_model": BASE_CHECKPOINT,
    "seed": SEED,
    "n_rich_examples": len(df_sc),
    "score_juez_mean": float(_scores.mean()) if len(_scores) else 0.0,
    "score_juez_std": float(_scores.std()) if len(_scores) else 0.0,
    "f1_exacto_mean": float(df_sc["f1_exacto"].mean()),
    "dimension_dominio": {
        "aciertos": int(_validos.sum()),
        "total": int(len(_validos)),
        "tasa": float(_validos.sum() / len(_validos)) if len(_validos) else 0.0,
    },
    "position_bias": {
        "protocolo": "pairwise A/B con intercambio de orden (S06 Lab B)",
        "n_pares": int(_n_pares),
        "n_empates": _n_emp,
        "tasa_sesgo_posicion": float(_n_emp / _n_pares) if _n_pares else 0.0,
        "mitigation": "veredicto robusto (solo gana si coincide en ambos ordenes); el score final NO se promedia con el test",
    },
    "length_bias": {
        "pairs_quality_wins": int(n_ok),
        "pairs_total": int(n_total),
        "pct_quality_wins": float(n_ok / n_total) if n_total > 0 else 0.0,
        "mitigation": "pares controlados: corta correcta vs. larga parcialmente correcta",
    },
    "auto_preference_bias": {
        "output_anonymized": True,
        "format_standardized": True,
        "cross_family_test": False,
        "note": "Cannot quantify without a judge from a different model family",
    },
    "pattern_distribution": df_sc["patron"].value_counts().to_dict(),
}

json_path = OUTPUT_DIR / "llm_judge_bias_summary.json"
with open(json_path, "w", encoding="utf-8") as f:
    _j.dump(bias_summary, f, indent=2, ensure_ascii=False)
print(f"Bias summary -> {json_path}")
print()
print("RESUMEN FINAL")
print(f"  Score juez (pointwise)       : {bias_summary['score_juez_mean']:.3f}")
print(f"  Aciertos de dominio          : {bias_summary['dimension_dominio']['aciertos']}/{bias_summary['dimension_dominio']['total']}")
print(f"  Sesgo posición (empates A/B) : {_n_emp}/{_n_pares} ({bias_summary['position_bias']['tasa_sesgo_posicion']:.1%})")
print(f"  Sesgo longitud (calidad gana): {n_ok}/{n_total}")

